# TAF Data

**Author**: Phillipe Lothaller  
**Date**: 04/05/2026

---

Terminal Aerodrome Forecasts (TAFs) provide predicted weather conditions for aerodromes over a defined time period. The TAF data is extracted from the KLM timestamped weather database. Forecasted values are matched against the creation time of a flight's flight plan.

In [1]:
import logging
import polars as pl
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# load plot styles
# plot style (contml unavailable)
# set_plot_style()  # contml unavailable

# clear existing logging handlers (prevents duplicate logs in notebooks)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# configure logging
logging.basicConfig(
    level=logging.INFO,  
    format='%(asctime)s - %(levelname)s - %(message)s'
)

## Extract KLM SQL TAF Data

In [2]:
from pathlib import Path
import pandas as pd

# Resolve project root
_here = Path.cwd()
_root = _here
for _ in range(6):
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent

PROCESSED_DATA_PATH = _root / "data" / "processed"
SQL_PATH = _root / "sql"

# Load cleaned flight data (filtered and cleaned in exploration.ipynb)
cleaned_pd = pd.read_parquet(PROCESSED_DATA_PATH / "klm_cleaned_data_2023_2026(0).parquet")
cleaned_df = pl.from_pandas(cleaned_pd)

logging.info(f"Loaded cleaned flight data: {cleaned_df.height} rows, {len(cleaned_df.columns)} columns")

# Load TAF SQL query from file
taf_sql_path = SQL_PATH / "extract_taf_data.sql"
klm_taf_query = taf_sql_path.read_text()
logging.info(f"Loaded TAF SQL query from {taf_sql_path}")

2026-05-14 16:56:38,297 - INFO - Loaded cleaned flight data: 490946 rows, 79 columns
2026-05-14 16:56:38,299 - INFO - Loaded TAF SQL query from c:\Users\k454047\OneDrive - Air France KLM\Desktop\THESIS\Code Test\sql\extract_taf_data.sql


In [3]:
from google.cloud import bigquery

# Derive date range from cleaned data
start_date = str(cleaned_df["fl_scheduled_departure_date_utc"].min())[:10]
end_date = str(cleaned_df["fl_scheduled_departure_date_utc"].max())[:10]
logging.info(f"Extracting TAF data for date range: {start_date} to {end_date}")

query = klm_taf_query.replace('{start_date}', start_date).replace('{end_date}', end_date)
client = bigquery.Client()
raw_klm_taf_df = pl.from_pandas(client.query(query).to_dataframe())
logging.info(f"Extracted KLM TAF data: {raw_klm_taf_df.height} rows")

# Inner-join with cleaned flight data — keeps only cleaned/filtered flights
# and adds fl_touchdown_delay_min and other flight features for later analysis
cleaned_join_cols = [
    "flight_leg_key", "fl_touchdown_delay_min",
    "fl_actual_in_block_time_utc", "fl_aircraft_type",
    "fl_great_circle_distance_nm", "fl_actual_trip_duration_min",
    "fl_arrival_delay_difference_min", "fl_season", "arrival_hour",
]
cleaned_join_cols = [c for c in cleaned_join_cols if c in cleaned_df.columns]

raw_klm_taf_df = raw_klm_taf_df.join(
    cleaned_df.select(cleaned_join_cols).rename({"flight_leg_key": "flight_key_id"}),
    on="flight_key_id",
    how="inner",
)

logging.info(f"After join with cleaned data: {raw_klm_taf_df.height} flights retained")
raw_klm_taf_df.head()

2026-05-14 16:56:52,137 - INFO - Extracting TAF data for date range: 2023-10-01 to 2026-03-31
c:\Users\k454047\OneDrive - Air France KLM\Desktop\THESIS\Code Test\.venv\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
c:\Users\k454047\OneDrive - Air France KLM\Desktop\THESIS\Code Test\.venv\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds.

flight_key_id,fl_scheduled_arrival_time_utc,fl_actual_arrival_time_utc,fl_departure_airport,fl_arrival_airport,lkpa_arrival_airport_icao_code,fpts_event_timestamp,fpts_scheduled_arrival_time,fpts_scheduled_departure_time,taf_report,taf_timestamp,has_taf_report,fl_touchdown_delay_min,fl_actual_in_block_time_utc,fl_aircraft_type,fl_great_circle_distance_nm,fl_actual_trip_duration_min,fl_arrival_delay_difference_min,fl_season,arrival_hour
str,"datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,datetime[μs],str,str,str,datetime[μs],i64,i64,datetime[μs],str,i64,i64,i64,str,i32
"""2023-10-01|KL|1263||AMS""",2023-10-01 20:25:00 UTC,2023-10-01 20:27:00 UTC,"""AMS""","""NCE""","""LFMN""",2023-10-01 16:18:59.211,"""2023-10-01T20:25Z""","""2023-10-01T18:30Z""","""TAF LFMN 011400 0115/0221 1800…",2023-10-01 14:00:00,1,2,2023-10-01 20:27:00,"""EMJ""",528,81,2,"""S23""",20
"""2023-10-02|KL|0867||AMS""",2023-10-03 03:40:00 UTC,2023-10-03 03:31:00 UTC,"""AMS""","""ICN""","""RKSI""",2023-10-02 14:32:08.068,"""2023-10-03T03:40Z""","""2023-10-02T16:00Z""","""TAF RKSI 021100 0212/0318 3600…",2023-10-02 11:00:00,1,-4,2023-10-03 03:31:00,"""787""",4619,661,-9,"""S23""",3
"""2023-10-04|KL|1515||AMS""",2023-10-04 08:00:00 UTC,2023-10-04 08:28:00 UTC,"""AMS""","""NWI""","""EGSH""",2023-10-04 07:00:54.457,"""2023-10-04T08:00Z""","""2023-10-04T07:10Z""","""TAF EGSH 040453 0406/0415 2400…",2023-10-04 04:53:00,1,-8,2023-10-04 08:28:00,"""EMJ""",129,34,28,"""S23""",8
"""2023-10-07|KL|1549||AMS""",2023-10-07 15:45:00 UTC,2023-10-07 15:50:00 UTC,"""AMS""","""LBA""","""EGNM""",2023-10-07 12:22:23.482,"""2023-10-07T15:45Z""","""2023-10-07T14:35Z""","""TAF EGNM 071211 0712/0812 2200…",2023-10-07 12:11:00,1,1,2023-10-07 15:50:00,"""EMJ""",249,48,5,"""S23""",15
"""2023-10-07|KL|1655||AMS""",2023-10-07 14:50:00 UTC,2023-10-07 14:41:00 UTC,"""AMS""","""VCE""","""LIPZ""",2023-10-07 10:32:01.707,"""2023-10-07T14:50Z""","""2023-10-07T13:05Z""","""TAF LIPZ 070500 0706/0812 VRB0…",2023-10-07 05:00:00,1,-9,2023-10-07 14:41:00,"""737""",505,77,-9,"""S23""",14


In [ ]:
n_flight_details = raw_klm_taf_df.select(pl.col("flight_key_id")).n_unique()
logging.info(f"Number of unique flights in the dataset: {n_flight_details}")

n_obserations_taf = raw_klm_taf_df.filter(pl.col("has_taf_report") == 1).height
logging.info(f"Number of observations with KLM TAF data available: {n_obserations_taf} ({n_obserations_taf / n_flight_details:.2%})")

## GE TAF Data Comparison

For this section we compare the TAF reports extracted from the KLM database, with the TAF reports stored in GE.

In [ ]:
# Load GE TAF data for the years 2023-2024
years = [2023, 2024]
ge_data_dir = _root / "data" / "external" / "ge_taf_data"

frames = []
for year in years:
    path = ge_data_dir / f"taf_data_{year}.csv"
    if not path.exists():
        logging.warning(f"GE TAF file not found: {path} — skipping year {year}")
        continue
    df = pl.read_csv(path, infer_schema=False)
    frames.append(df)

if frames:
    ge_taf_df = pl.concat(frames, how="diagonal_relaxed")
    logging.info(f"Loaded GE TAF data: {ge_taf_df.height} rows")
    ge_taf_df.head()
else:
    ge_taf_df = None
    logging.warning("No GE TAF data found. GE comparison section will be skipped.")

In [ ]:
# add TAF to the beiginning of the TAF message if it does not already start with TAF
ge_taf_df = ge_taf_df.with_columns(
    pl.when(pl.col("TAF message").is_null())
    .then(None)
    .when(pl.col("TAF message").str.starts_with("TAF"))
    .then(pl.col("TAF message"))
    .otherwise(pl.concat_str([pl.lit("TAF "), pl.col("TAF message")]))
    .alias("taf_report")
)

In [ ]:
ge_taf_df = ge_taf_df.with_columns(
    [
        pl.col("Flight Date (UTC) | Out")
        .str.to_datetime("%m/%d/%Y %H:%M", strict=False)
        .dt.date()
        .cast(pl.Utf8)
        .alias("headStationDepartureDateUtc"),

        pl.col("Flight Number")
        .str.strip_chars()
        .str.extract(r"(\d+)", 1)
        .str.zfill(4)
        .alias("flightNumberPadded"),

        pl.lit("KL").alias("carrier"),
        pl.lit("").alias("suffix"),
        pl.col("Origin").str.strip_chars().alias("departureAirport"),
    ]
).with_columns(
    pl.concat_str(
        [
            pl.col("headStationDepartureDateUtc"),
            pl.col("carrier"),
            pl.col("flightNumberPadded"),
            pl.col("suffix"),
            pl.col("departureAirport"),
        ],
        separator="|",
    ).alias("flight_key_id")
)

ge_taf_df.select(
    [
        "Flight Date (UTC) | Out",
        "Flight Number",
        "Origin",
        "flight_key_id",
        "TAF message",
    ]
).head()

In [ ]:
if ge_taf_df is not None:
    # Join KLM and GE TAF data on flight_key_id
    taf_df = (
        raw_klm_taf_df.select(["flight_key_id", "fl_scheduled_arrival_time_utc", "fl_touchdown_delay_min", "taf_report"])
        .rename({"taf_report": "klm_taf_report", "fl_scheduled_arrival_time_utc": "klm_arrival_timestamp"})
        .join(
            ge_taf_df.select(["flight_key_id", "Flight Date (UTC) | On", "TAF message"])
            .with_columns(
                pl.col("Flight Date (UTC) | On")
                .str.to_datetime("%m/%d/%Y %H:%M", strict=False)
                .alias("ge_arrival_timestamp")
            )
            .drop("Flight Date (UTC) | On")
            .rename({"TAF message": "ge_taf_report"}),
            on="flight_key_id",
            how="full",
            coalesce=True,
        )
    )
else:
    # No GE data — use only KLM TAF
    taf_df = (
        raw_klm_taf_df.select(["flight_key_id", "fl_scheduled_arrival_time_utc", "fl_touchdown_delay_min", "taf_report"])
        .rename({"taf_report": "klm_taf_report", "fl_scheduled_arrival_time_utc": "klm_arrival_timestamp"})
        .with_columns([
            pl.lit(None).cast(pl.Utf8).alias("ge_taf_report"),
            pl.lit(None).cast(pl.Datetime).alias("ge_arrival_timestamp"),
        ])
    )

n_observations_comparison = taf_df.height
n_observations_klm = taf_df.filter(pl.col("klm_taf_report").is_not_null()).height
n_observations_ge = taf_df.filter(pl.col("ge_taf_report").is_not_null()).height
logging.info(f"Number of observations in comparison: {n_observations_comparison}")
logging.info(f"Number of observations with KLM TAF: {n_observations_klm}")
logging.info(f"Number of observations with GE TAF: {n_observations_ge}")

In [ ]:
def normalize_taf_report_expr(report_col: str) -> pl.Expr:
    return (
        pl.when(pl.col(report_col).is_null())
        .then(None)
        .otherwise(
            pl.col(report_col)
            .cast(pl.Utf8)
            .str.to_uppercase()
            .str.replace(r"^TAF\s+", "")
            .str.replace(r"^([A-Z]{4})\s+\d{6}Z?\b", r"${1}")
            .str.replace_all(r"\s+", " ")
            .str.strip_chars()
        )
    )

taf_df = taf_df.with_columns(
    [
        normalize_taf_report_expr("klm_taf_report").alias("normalized_klm_taf_report"),
        normalize_taf_report_expr("ge_taf_report").alias("normalized_ge_taf_report")
    ]
)

have_same_taf = taf_df.filter(
    (pl.col("normalized_klm_taf_report") == pl.col("normalized_ge_taf_report"))
    & pl.col("normalized_klm_taf_report").is_not_null()
).height
logging.info(f"Number of observations with the same TAF report in both datasets: {have_same_taf} ({have_same_taf / n_observations_comparison:.2%})")

## Machine Learning Feature Extraction

The Metafora (https://github.com/ramondalmau/metafora) python library will be used to extract useful machine learning features from the different TAF messages.

In [ ]:
from weather.extract_features import extract_ml_features_from_weather_report

taf_features_df, _ = extract_ml_features_from_weather_report(
    weather_df=taf_df,
    report_col="klm_taf_report",
    timestamp_col="klm_arrival_timestamp",
    report_type="TAF",
)

taf_features_df, _ = extract_ml_features_from_weather_report(
    weather_df=taf_features_df,
    report_col="ge_taf_report",
    timestamp_col="ge_arrival_timestamp",
    report_type="TAF",
)

taf_features_df.head()

In [ ]:
# Plot comparison of wind speed distributions from KLM and GE TAF reports
def __plot_feature_comparison(df, col_a, col_b, label_a, label_b, title, distribution_xlabel, color_index=0):
    import numpy as _np
    colors = ['#4C72B0', '#DD8452', '#2ca02c', '#9467bd', '#8c564b']
    vals_a = df[col_a].drop_nulls().to_numpy() if col_a in df.columns else _np.array([])
    vals_b = df[col_b].drop_nulls().to_numpy() if col_b in df.columns else _np.array([])
    if not len(vals_a) and not len(vals_b):
        print(f'Columns {col_a!r} / {col_b!r} not found — skipping plot.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for i, (ax, vals, label) in enumerate(zip(axes, [vals_a, vals_b], [label_a, label_b])):
        if not len(vals):
            continue
        c = colors[(color_index + i) % len(colors)]
        ax.hist(vals, bins=50, alpha=0.7, color=c)
        ax.axvline(float(vals.mean()), color='red', linewidth=1.5,
                   label=f'Mean: {float(vals.mean()):.2f}')
        ax.set_xlabel(distribution_xlabel)
        ax.set_ylabel('Frequency')
        ax.set_title(label)
        ax.legend(fontsize=8)
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

__plot_feature_comparison(
    df=taf_features_df,
    col_a="klm_taf_wind_speed",
    col_b="ge_taf_wind_speed",
    label_a = "KLM TAF Wind Speed",
    label_b = "GE TAF Wind Speed",
    title="Comparison of Wind Speed (kt) from KLM and GE TAF Reports at ATA-0",
    distribution_xlabel="Wind Speed (kt)",
    color_index=0
)

In [ ]:
__plot_feature_comparison(
    df=taf_features_df,
    col_a="klm_taf_visibility_distance",
    col_b="ge_taf_visibility_distance",
    label_a = "KLM TAF Visibility Distance",
    label_b = "GE TAF Visibility Distance",
    title="Comparison of Visibility Distance (m) from KLM and GE TAF Reports at ATA-0",
    distribution_xlabel="Visibility Distance (m)",
    color_index=0
)

In [ ]:
__plot_feature_comparison(
    df=taf_features_df,
    col_a="klm_taf_clouds_height",
    col_b="ge_taf_clouds_height",
    label_a = "KLM TAF Clouds Height",
    label_b = "GE TAF Clouds Height",
    title="Comparison of Clouds Height (m) from KLM and GE TAF Reports at ATA-0",
    distribution_xlabel="Clouds Height (m)",
    color_index=0
)

## TAF vs. METAR Data Comparison

In [ ]:
# Load processed METAR data if available; otherwise skip the METAR-TAF comparison
metar_data_path = PROCESSED_DATA_PATH / "metar_features.parquet"
if metar_data_path.exists():
    metar_df = pl.read_parquet(metar_data_path)
    logging.info(f"Loaded METAR features data: {metar_df.height} rows")
else:
    # Fall back to cleaned flight data with flight_key_id mapping
    logging.warning(f"Processed METAR data not found at {metar_data_path}. "
                    "Run metar_data.ipynb first and save the output, "
                    "or the METAR vs TAF comparison section will be skipped.")
    metar_df = None

In [ ]:
from weather.extract_features import extract_ml_features_from_weather_report

if metar_df is not None:
    metar_features_df, _ = extract_ml_features_from_weather_report(
        weather_df=metar_df,
        report_col="klm_t0_metar_report",
        timestamp_col="klm_t0_metar_obs_timestamp",
        insert_before_col="has_klm_t0_metar_report",
        report_type="METAR"
    )

    metar_features_df, _ = extract_ml_features_from_weather_report(
        weather_df=metar_features_df,
        report_col="klm_t30_metar_report",
        timestamp_col="klm_t30_metar_obs_timestamp",
        insert_before_col="has_klm_t30_metar_report",
        report_type="METAR"
    )
else:
    metar_features_df = None
    logging.warning("METAR features not available — skipping METAR feature extraction.")


In [ ]:
if metar_features_df is not None:
    # Join metar_features_df with taf_features_df on flight_key_id
    weather_features_df = taf_features_df.join(
        metar_features_df,
        on="flight_key_id",
        how="inner"
    )
    logging.info(f"Combined METAR + TAF features: {weather_features_df.height} rows")
    weather_features_df.head()
else:
    weather_features_df = taf_features_df
    logging.warning("METAR features not available — weather_features_df contains TAF only.")

In [ ]:
# Local replacement for contml.utils.plotting.plot_feature_comparison
def __plot_feature_comparison(df, col_a, col_b, label_a, label_b, title, distribution_xlabel, color_index=0):
    import numpy as _np
    colors = ['#4C72B0', '#DD8452', '#2ca02c', '#9467bd', '#8c564b']
    vals_a = df[col_a].drop_nulls().to_numpy() if col_a in df.columns else _np.array([])
    vals_b = df[col_b].drop_nulls().to_numpy() if col_b in df.columns else _np.array([])
    if not len(vals_a) and not len(vals_b):
        print(f'Columns {col_a!r} / {col_b!r} not found — skipping plot.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for i, (ax, vals, label) in enumerate(zip(axes, [vals_a, vals_b], [label_a, label_b])):
        if not len(vals):
            continue
        c = colors[(color_index + i) % len(colors)]
        ax.hist(vals, bins=50, alpha=0.7, color=c)
        ax.axvline(float(vals.mean()), color='red', linewidth=1.5,
                   label=f'Mean: {float(vals.mean()):.2f}')
        ax.set_xlabel(distribution_xlabel)
        ax.set_ylabel('Frequency')
        ax.set_title(label)
        ax.legend(fontsize=8)
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

__plot_feature_comparison(
    df=weather_features_df,
    col_a="tsw_t0_metar_wind_speed",
    col_b="klm_taf_wind_speed",
    label_a = "TSW METAR Wind Speed",
    label_b = "KLM TAF Wind Speed",
    title="Comparison of Wind Speed (kt) from TSW METAR and KLM TAF Reports at ATA-0",
    distribution_xlabel="Wind Speed (kt)",
    color_index=0
)

In [ ]:
__plot_feature_comparison(
    df=weather_features_df,
    col_a="tsw_t0_metar_visibility_distance",
    col_b="klm_taf_visibility_distance",
    label_a = "TSW METAR Visibility Distance",
    label_b = "KLM TAF Visibility Distance",
    title="Comparison of Visibility Distance (m) from TSW METAR and KLM TAF Reports at ATA-0",
    distribution_xlabel="Visibility Distance (m)",
    color_index=2
)

In [ ]:
__plot_feature_comparison(
    df=weather_features_df,
    col_a="tsw_t0_metar_clouds_height",
    col_b="klm_taf_clouds_height",
    label_a = "TSW METAR Cloud Ceiling Height",
    label_b = "KLM TAF Cloud Ceiling Height",
    title="Comparison of Cloud Ceiling Height (m) from TSW METAR and KLM TAF Reports at ATA-0",
    distribution_xlabel="Cloud Ceiling Height (m)",
    color_index=4
)


## Missing Data Analysis

Examine how many flights are missing TAF reports and whether missingness is concentrated in specific airports or time periods.


In [ ]:
import numpy as np

n_total = taf_features_df.height

# Overall availability for KLM and GE TAF
has_klm = taf_features_df.filter(pl.col("klm_taf_report").is_not_null()).height
has_ge  = taf_features_df.filter(pl.col("ge_taf_report").is_not_null()).height if "ge_taf_report" in taf_features_df.columns else 0

logging.info(f"Total flights: {n_total}")
logging.info(f"KLM TAF available: {has_klm} ({has_klm / n_total:.2%})")
logging.info(f"GE TAF available: {has_ge} ({has_ge / n_total:.2%})")

fig, ax = plt.subplots(figsize=(7, 4))
sources = ["KLM TAF", "GE TAF"]
counts = [has_klm, has_ge]
bars = ax.bar(sources, [c / n_total * 100 for c in counts], color=["#4C72B0", "#DD8452"])
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{c / n_total:.1%}", ha="center", fontsize=11)
ax.set_ylim(0, 115)
ax.set_ylabel("Availability (%)")
ax.set_title("TAF Report Availability by Source")
plt.tight_layout()
plt.show()

In [ ]:
# Missing TAF by arrival airport — using the full raw_klm_taf_df which has lkpa_arrival_airport_icao_code
if "lkpa_arrival_airport_icao_code" in raw_klm_taf_df.columns:
    missing_taf_by_airport = (
        raw_klm_taf_df
        .group_by("lkpa_arrival_airport_icao_code")
        .agg(
            pl.len().alias("n_flights"),
            (pl.col("has_taf_report") == 1).sum().alias("n_taf"),
        )
        .with_columns(
            (1 - pl.col("n_taf") / pl.col("n_flights")).alias("pct_missing")
        )
        .filter(pl.col("n_flights") >= 50)
        .sort("pct_missing", descending=True)
        .head(20)
    )

    airports = missing_taf_by_airport["lkpa_arrival_airport_icao_code"].to_list()
    pct_miss = (missing_taf_by_airport["pct_missing"] * 100).to_list()

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(range(len(airports)), pct_miss, color="#4C72B0", alpha=0.8)
    ax.set_xticks(range(len(airports)))
    ax.set_xticklabels(airports, rotation=45, ha="right")
    ax.set_ylabel("Missing TAF (%)")
    ax.set_title("KLM TAF Missing Data by Airport (Top 20, ≥50 flights)")
    plt.tight_layout()
    plt.show()


## Outlier Detection

Identify extreme or implausible values in TAF-derived weather features.


In [ ]:
taf_feature_cols = [
    ("klm_taf_wind_speed",          "KLM TAF Wind Speed (kt)"),
    ("klm_taf_visibility_distance", "KLM TAF Visibility (m)"),
    ("klm_taf_clouds_height",       "KLM TAF Cloud Height (m)"),
]
taf_feature_cols = [(c, l) for c, l in taf_feature_cols if c in taf_features_df.columns]

if taf_feature_cols:
    fig, axes = plt.subplots(1, len(taf_feature_cols), figsize=(5 * len(taf_feature_cols), 5))
    if len(taf_feature_cols) == 1:
        axes = [axes]

    for ax, (col, label) in zip(axes, taf_feature_cols):
        values = taf_features_df.select(pl.col(col)).drop_nulls().to_series().to_numpy()
        ax.boxplot(values, vert=True, patch_artist=True,
                   boxprops=dict(facecolor="#DD8452", alpha=0.6))
        q1, q3 = np.percentile(values, [25, 75])
        iqr = q3 - q1
        n_outliers = int(((values < q1 - 3 * iqr) | (values > q3 + 3 * iqr)).sum())
        ax.set_title(f"{label}\nn={len(values):,}  outliers (3×IQR): {n_outliers}")
        ax.set_xticks([])

    plt.suptitle("KLM TAF Feature Distributions", fontsize=12)
    plt.tight_layout()
    plt.show()

# Descriptive statistics
outlier_rows = []
all_taf_cols = [
    ("klm_taf_wind_speed",          "KLM TAF Wind Speed"),
    ("ge_taf_wind_speed",           "GE TAF Wind Speed"),
    ("klm_taf_visibility_distance", "KLM TAF Visibility"),
    ("ge_taf_visibility_distance",  "GE TAF Visibility"),
    ("klm_taf_clouds_height",       "KLM TAF Cloud Height"),
    ("ge_taf_clouds_height",        "GE TAF Cloud Height"),
]
for col, label in all_taf_cols:
    if col not in taf_features_df.columns:
        continue
    values = taf_features_df.select(pl.col(col)).drop_nulls().to_series().to_numpy()
    if len(values) == 0:
        continue
    q1, q3 = np.percentile(values, [25, 75])
    iqr = q3 - q1
    outlier_rows.append({
        "Feature": label,
        "n_valid": len(values),
        "min": round(float(values.min()), 2),
        "p1": round(float(np.percentile(values, 1)), 2),
        "median": round(float(np.median(values)), 2),
        "p99": round(float(np.percentile(values, 99)), 2),
        "max": round(float(values.max()), 2),
        "n_outliers_3iqr": int(((values < q1 - 3 * iqr) | (values > q3 + 3 * iqr)).sum()),
    })

if outlier_rows:
    print(pl.DataFrame(outlier_rows))


## METAR vs TAF Feature Comparison

Compare actual weather conditions (from METAR observations) against the forecasted values (from TAF reports) for the same flights. Differences indicate the forecast error at the time of arrival.


In [ ]:
if metar_features_df is not None and "weather_features_df" in dir() and weather_features_df is not None:
    # Feature pairs: (METAR actual col, TAF forecast col, label)
    comparison_pairs = [
        ("klm_t0_metar_wind_speed",          "klm_taf_wind_speed",          "Wind Speed (kt)"),
        ("klm_t0_metar_visibility_distance", "klm_taf_visibility_distance", "Visibility (m)"),
        ("klm_t0_metar_clouds_height",       "klm_taf_clouds_height",       "Cloud Height (m)"),
    ]
    available_pairs = [
        (m, t, l) for m, t, l in comparison_pairs
        if m in weather_features_df.columns and t in weather_features_df.columns
    ]

    if available_pairs:
        fig, axes = plt.subplots(1, len(available_pairs), figsize=(6 * len(available_pairs), 5))
        if len(available_pairs) == 1:
            axes = [axes]

        for ax, (metar_col, taf_col, label) in zip(axes, available_pairs):
            sub = weather_features_df.select([metar_col, taf_col]).drop_nulls().to_pandas()
            ax.scatter(sub[metar_col], sub[taf_col], alpha=0.1, s=3, rasterized=True)
            # Perfect agreement line
            lims = [min(sub[metar_col].min(), sub[taf_col].min()),
                    max(sub[metar_col].max(), sub[taf_col].max())]
            ax.plot(lims, lims, color="red", linestyle="--", linewidth=1, label="Perfect forecast")
            r = sub[metar_col].corr(sub[taf_col])
            bias = (sub[taf_col] - sub[metar_col]).mean()
            mae = (sub[taf_col] - sub[metar_col]).abs().mean()
            ax.set_xlabel(f"METAR Actual — {label}")
            ax.set_ylabel(f"TAF Forecast — {label}")
            ax.set_title(f"r={r:.3f}  bias={bias:.2f}  MAE={mae:.2f}")
            ax.legend(fontsize=8)

        plt.suptitle("METAR Actual vs KLM TAF Forecast (matched flights)", fontsize=12)
        plt.tight_layout()
        plt.show()

        # Distribution of forecast errors
        fig, axes = plt.subplots(1, len(available_pairs), figsize=(6 * len(available_pairs), 4))
        if len(available_pairs) == 1:
            axes = [axes]

        for ax, (metar_col, taf_col, label) in zip(axes, available_pairs):
            sub = weather_features_df.select([metar_col, taf_col]).drop_nulls().to_pandas()
            errors = sub[taf_col] - sub[metar_col]
            ax.hist(errors, bins=50, alpha=0.7, color="#4C72B0", edgecolor="none")
            ax.axvline(errors.mean(), color="red", linewidth=1.5, label=f"Mean bias: {errors.mean():.2f}")
            ax.set_xlabel(f"TAF − METAR ({label})")
            ax.set_ylabel("Frequency")
            ax.set_title(f"Forecast Error: {label}")
            ax.legend(fontsize=8)

        plt.suptitle("Distribution of TAF Forecast Errors vs METAR Observations", fontsize=12)
        plt.tight_layout()
        plt.show()
else:
    logging.warning("METAR features not available — METAR vs TAF comparison skipped.")


## Weather–Delay Correlations

Examine how TAF-forecasted weather at the time of flight plan creation correlates with the actual touchdown delay.


In [ ]:
import pandas as pd
import seaborn as sns

# TAF weather features and target
taf_corr_cols = [
    "fl_touchdown_delay_min",
    "klm_taf_wind_speed",
    "klm_taf_visibility_distance",
    "klm_taf_clouds_height",
    "ge_taf_wind_speed",
    "ge_taf_visibility_distance",
    "ge_taf_clouds_height",
]
taf_corr_cols = [c for c in taf_corr_cols if c in taf_features_df.columns]

if "fl_touchdown_delay_min" in taf_corr_cols and len(taf_corr_cols) > 1:
    corr_pd = taf_features_df.select(taf_corr_cols).to_pandas().astype(float)
    corr_matrix = corr_pd.corr()

    fig, ax = plt.subplots(figsize=(max(7, len(taf_corr_cols)), max(5, len(taf_corr_cols) - 2)))
    sns.heatmap(
        corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
        center=0, linewidths=0.5, ax=ax, annot_kws={"size": 9}
    )
    ax.set_title("Pearson Correlation Matrix — TAF Features vs Touchdown Delay")
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.yticks(fontsize=9)
    plt.tight_layout()
    plt.show()

    # Scatter plots for top TAF weather features vs delay
    scatter_pairs = [(c, c.replace("_", " ").title()) for c in taf_corr_cols if c != "fl_touchdown_delay_min"][:3]
    if scatter_pairs:
        fig, axes = plt.subplots(1, len(scatter_pairs), figsize=(6 * len(scatter_pairs), 5))
        if len(scatter_pairs) == 1:
            axes = [axes]
        plot_pd = corr_pd.dropna()
        for ax, (col, label) in zip(axes, scatter_pairs):
            ax.scatter(plot_pd[col], plot_pd["fl_touchdown_delay_min"],
                       alpha=0.1, s=2, rasterized=True)
            mask = plot_pd[col].notna() & plot_pd["fl_touchdown_delay_min"].notna()
            if mask.sum() > 10:
                z = np.polyfit(plot_pd.loc[mask, col], plot_pd.loc[mask, "fl_touchdown_delay_min"], 1)
                p = np.poly1d(z)
                x_line = np.linspace(plot_pd[col].min(), plot_pd[col].max(), 100)
                ax.plot(x_line, p(x_line), color="red", linewidth=1.5, label="Trend")
            r = plot_pd[col].corr(plot_pd["fl_touchdown_delay_min"])
            ax.set_xlabel(label)
            ax.set_ylabel("Touchdown Delay (min)")
            ax.set_title(f"r = {r:.3f}")
            ax.legend()
        plt.suptitle("TAF Features vs Touchdown Delay", fontsize=12)
        plt.tight_layout()
        plt.show()
else:
    logging.warning("fl_touchdown_delay_min not available in taf_features_df — correlation analysis skipped.")